# 第三讲：随机数生成与蒙特卡洛模拟**学习目标**- 理解「随机数」在量化中的角色：模拟未来、测试策略、评估风险- 掌握新一代 Generator API，告别过时的写法- 熟练使用 `uniform`、`normal`、`integers` 三大常用分布- 理解随机种子与可复现性的底层逻辑- 用蒙特卡洛方法解决两个经典问题：掷骰子实验 + 估算 π### 什么是蒙特卡洛？蒙特卡洛不是一种算法，而是一种**思想**：用大量随机试验来逼近一个确定性的答案。比如你不知道 π 的值，但你可以往正方形里撒 10000 个点，数圆内的比例，就能逼近 π。撒的点越多，结果越准——这就是大数定律。量化分析里无处不在：模拟股价走势、评估策略风险、定价衍生品……全部基于这个思想。---

In [ ]:
# ─── 导入工具箱 ───import numpy as npimport matplotlib.pyplot as plt# matplotlib 是 Python 最常用的画图库，我们用它来可视化随机分布# 设置中文字体 —— 否则图表里的中文会变成方块# macOS 用 'Heiti SC'，Windows 用 'SimHei'plt.rcParams['font.sans-serif'] = ['Heiti SC', 'SimHei', 'Arial Unicode MS']plt.rcParams['axes.unicode_minus'] = False  # 让负号正常显示print(f"NumPy 版本: {np.__version__}")

---## 3.1 Generator API —— 新一代随机数引擎### 先理解一个反直觉的概念：「随机数」其实是确定的计算机不会「真随机」，它用的是一种数学公式，你给它一个**种子（seed）**，它就输出一串看起来随机的数字。同样的种子永远产出同样的序列——这不是 bug，这是**科学计算的核心需求：实验必须可复现**。### 新旧对比```旧写法:  np.random.seed(42);  np.random.rand(3)     # 全局状态，多线程不安全新写法:  rng = np.random.default_rng(42);  rng.random(3)   # 每个 rng 独立，更安全```类比：旧写法是全家人共用一个电视遥控器（互相干扰），新写法是每人一个遥控器。

In [ ]:
# ─── 创建随机数生成器 ───# default_rng() 是 NumPy 推荐的入口，内部自动选择最好的算法（PCG64）rng = np.random.default_rng(seed=42)# seed=42 可以换成任何整数。42 是科幻小说《银河系漫游指南》的梗，程序员常用它# 生成 5 个 [0, 1) 之间的随机小数# [0, 1) 表示 >=0 且 <1 —— 包含 0，不包含 1print("rng.random(5):", rng.random(5))print()# ─── 验证可复现性：同样的种子 = 同样的结果 ───rng2 = np.random.default_rng(seed=42)print("rng2.random(5):", rng2.random(5))print("完全相同?", np.array_equal(rng.random(5), rng2.random(5)))# 如果是 True，说明种子确实控制了随机序列

In [ ]:
# ─── 不同种子 = 不同序列 ───seeds = [42, 123, 2024]for s in seeds:    rng = np.random.default_rng(seed=s)    print(f"seed={s:>4} -> 前 5 个随机数: {rng.random(5)}")# 每个种子产生了不同的序列# 但如果你再次运行这个 cell，结果完全一样！print()# 再次用 seed=42，验证结果一致rng = np.random.default_rng(seed=42)print(f"seed=42  再次确认:   {rng.random(5)}")# 所以你可以把种子告诉别人，他们就能完全复现你的分析

---## 3.2 uniform —— 均匀分布最基础的分布。每个值在指定区间内「等可能」出现，就像在区间里随机抓一个数。```pythonrng.uniform(low=0.0, high=1.0, size=None)```| 参数 | 含义 | 默认值 ||------|------|--------|| `low` | 下界（包含） | 0.0 || `high` | 上界（不包含） | 1.0 || `size` | 输出形状：一个整数 = 一维数组；元组 = 多维数组 | None（返回单个值） |> ⚠ **常见坑：** `high` 是不包含的。`uniform(0, 1, 10)` 生成的数永远 < 1，不会等于 1。

In [ ]:
rng = np.random.default_rng(seed=42)# 10 个 [0, 1) 的随机数print("[0, 1):   ", rng.uniform(0, 1, 10))# 5 个 [-5, 5) 的随机数print("[-5, 5):  ", rng.uniform(-5, 5, 5))# 3x4 矩阵，范围 [10, 20) —— 常用于生成模拟数据print("\n3x4 矩阵 [10, 20):")print(rng.uniform(10, 20, size=(3, 4)))# size=(3, 4) 生成 3 行 4 列。没写 size 的话只返回一个数字

### 可视化：均匀分布长什么样？画 100,000 个样本的直方图。理论上每个区间的高度应该一样（因为是「均匀」的）。

In [ ]:
# ─── 可视化均匀分布 ───rng = np.random.default_rng(seed=123)samples = rng.uniform(0, 1, 100_000)  # 10万个样本# Python 中 100_000 和 100000 完全一样，下划线只是为了阅读方便fig, ax = plt.subplots(figsize=(10, 5))# plt.subplots() 创建"画布(fig)"和"坐标轴(ax)"，一次调用搞定ax.hist(samples, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='white')# hist 画直方图。bins=50 分成 50 个柱子# density=True 让 y 轴显示概率密度（而不是计数），这样红线的 1.0 才能对应上ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='理论概率密度 = 1.0')# axhline = Axis Horizontal Line，画一条水平参考线ax.set_title('均匀分布 U(0,1) - 100,000 个样本', fontsize=14)ax.set_xlabel('值')ax.set_ylabel('概率密度')ax.legend()plt.tight_layout()  # 自动调整间距，避免标签被切掉plt.show()

---## 3.3 normal —— 正态分布（高斯分布）自然界和金融市场里**最常见的分布**。股价的日收益率、测量误差、身高体重……都近似正态。```pythonrng.normal(loc=0.0, scale=1.0, size=None)```| 参数 | 含义 | 默认值 ||------|------|--------|| `loc` | 均值 μ（分布的中心，钟形曲线的最高点） | 0.0 || `scale` | 标准差 σ（曲线的"胖瘦"，越大越扁平） | 1.0 |### 68-95-99.7 法则- 68% 的数据落在 μ ± σ 内- 95% 落在 μ ± 2σ 内- 99.7% 落在 μ ± 3σ 内这在量化里非常有用：如果你算出了一只股票日收益率的均值和标准差，就可以估计极端波动的概率。

In [ ]:
rng = np.random.default_rng(seed=42)# 标准正态分布 N(0, 1)：均值=0，标准差=1print("标准正态 N(0,1):", rng.normal(0, 1, 5))# 均值为 5，标准差为 2：大部分值会在 5-4=1 到 5+4=9 之间print("N(5, 2):", rng.normal(5, 2, 5))# 2x3 矩阵print("\n2x3 矩阵 N(100, 15):")print(rng.normal(100, 15, size=(2, 3)))

In [ ]:
# ─── 可视化正态分布：直方图 + 理论曲线 ───rng = np.random.default_rng(seed=42)samples = rng.normal(loc=0, scale=1, size=100_000)fig, ax = plt.subplots(figsize=(10, 5))ax.hist(samples, bins=80, density=True, alpha=0.7, color='steelblue', edgecolor='white')# 叠加理论正态分布曲线（红色）x = np.linspace(-4, 4, 200)  # 在 -4 到 4 之间取 200 个点画平滑曲线# 下面的公式是标准正态分布的概率密度函数（PDF）pdf = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)ax.plot(x, pdf, color='red', linewidth=2, label='理论 N(0,1) 曲线')ax.set_title('标准正态分布 N(0,1) - 100,000 个样本', fontsize=14)ax.set_xlabel('值')ax.set_ylabel('概率密度')ax.legend()plt.tight_layout()plt.show()# 验证 68-95-99.7 法则for k, label in [(1, "68%"), (2, "95%"), (3, "99.7%")]:    pct = np.mean(np.abs(samples) < k) * 100    print(f"  |x| < {k}: {pct:.1f}%  (理论: {label})")

---## 3.4 integers —— 随机整数模拟掷骰子、随机抽样、选股编号等离散场景。```pythonrng.integers(low, high=None, size=None, endpoint=False)```| 参数 | 含义 ||------|------|| `low` | 下界（包含） || `high` | 上界。默认不包含，除非 `endpoint=True` || `endpoint` | 如果为 True，`high` 也被包含 |> ⚠ **常见坑：** 默认 `endpoint=False`，所以 `integers(1, 7)` 生成的是 1~6，不是 1~7。想包含 7 要写 `endpoint=True`。

In [ ]:
rng = np.random.default_rng(seed=42)# [0, 10) 的随机整数 —— 不包含 10print("[0, 10): ", rng.integers(0, 10, 10))# [1, 7) -> 模拟掷骰子：1 到 6（因为不包含 7）print("掷 5 次骰子:", rng.integers(1, 7, 5))# endpoint=True：包含上限print("[1, 6] 含两端:", rng.integers(1, 6, 10, endpoint=True))# 3x4 矩阵，范围 [0, 100)print("\n3x4 矩阵 [0, 100):")print(rng.integers(0, 100, size=(3, 4)))

---## 3.5 其他常用分布速查| 方法 | 分布 | 量化用途 ||------|------|----------|| `rng.uniform(low, high)` | 均匀分布 | 等概率随机抽样 || `rng.normal(loc, scale)` | 正态分布 | 收益率建模、VaR 计算 || `rng.integers(low, high)` | 随机整数 | 选股编号、掷骰子 || `rng.exponential(scale)` | 指数分布 | 订单到达时间间隔 || `rng.poisson(lam)` | 泊松分布 | 日内交易次数 || `rng.binomial(n, p)` | 二项分布 | N 次试验中成功次数 || `rng.choice(array, size)` | 随机抽样 | 从候选池中抽股票 |---

## 3.6 蒙特卡洛实战 ①：模拟掷骰子 10,000 次**问题：** 投掷公平六面骰子 10,000 次，各点数出现频率是否接近 1/6？**蒙特卡洛思路：** 用 `integers(1, 7, 10000)` 一次性生成 10000 次投掷结果，然后统计频率。

In [ ]:
rng = np.random.default_rng(seed=42)n_trials = 10_000# 一次性模拟 10,000 次掷骰子 —— 这就是向量化的威力dice_rolls = rng.integers(1, 7, size=n_trials)print(f"前 20 次结果: {dice_rolls[:20]}")# 理论平均值 = (1+2+3+4+5+6)/6 = 3.5print(f"平均值: {dice_rolls.mean():.3f} (理论值: 3.5)")print(f"标准差: {dice_rolls.std():.3f} (理论值: 1.708)")

In [ ]:
# ─── 统计每个点数出现次数 ───faces, counts = np.unique(dice_rolls, return_counts=True)# np.unique 返回"去重后的值"和"每个值出现的次数"print("点数频率分布:")print("-" * 35)for face, count in zip(faces, counts):    freq = count / n_trials    bar = '|' * int(freq * 200)  # 用竖线画一个简单的条形图    print(f"  点数 {face}: {count:>5} 次  ({freq:.4f})  | 理论: 1/6 = {1/6:.4f}  {bar}")print("-" * 35)print(f"  总计:     {counts.sum():>5} 次")

In [ ]:
# ─── 绘制频率分布直方图 ───fig, axes = plt.subplots(1, 2, figsize=(14, 5))# 1 行 2 列：左边频数，右边频率# 左图：频数柱状图colors = ['#FF6B6B', '#FFA94D', '#FFD43B', '#69DB7C', '#4DABF7', '#DA77F2']axes[0].bar(faces, counts, color=colors, edgecolor='white', linewidth=1.5)axes[0].axhline(y=n_trials/6, color='black', linestyle='--', linewidth=1.5,                label=f'理论值: {n_trials/6:.0f} 次')axes[0].set_title(f'掷骰子 {n_trials:,} 次 - 频数统计', fontsize=13, fontweight='bold')axes[0].set_xlabel('点数')axes[0].set_ylabel('出现次数')axes[0].set_xticks(faces)axes[0].legend()# 右图：频率直方图 + 理论概率线frequencies = counts / n_trialsaxes[1].bar(faces, frequencies, color=colors, edgecolor='white', linewidth=1.5)axes[1].axhline(y=1/6, color='black', linestyle='--', linewidth=1.5,                label=f'理论概率: 1/6 = {1/6:.3f}')axes[1].set_title(f'掷骰子 {n_trials:,} 次 - 频率分布', fontsize=13, fontweight='bold')axes[1].set_xlabel('点数')axes[1].set_ylabel('频率')axes[1].set_xticks(faces)axes[1].set_ylim(0, 0.25)axes[1].legend()plt.tight_layout()plt.show()# 最大偏差：数据和理论的差距max_deviation = np.max(np.abs(frequencies - 1/6))print(f"\n最大偏差: {max_deviation:.4f} ({max_deviation / (1/6) * 100:.1f}%)")# 偏差会随着试验次数增加而减小 —— 大数定律

### 收敛性验证：大数定律的直观展示随着试验次数增加，频率会越来越接近理论值 1/6。这就是**大数定律**：样本越多，统计量越稳定。

In [ ]:
rng = np.random.default_rng(seed=123)n_max = 50_000rolls = rng.integers(1, 7, size=n_max)# 逐步计算累计频率：第一次投掷后的频率，前两次后的频率，...cumulative_freq = np.zeros((n_max, 6))for i in range(6):    # np.cumsum 计算累积和：第 n 个位置 = 前 n 次中点数 i+1 出现了多少次    cumulative_freq[:, i] = np.cumsum(rolls == (i + 1)) / np.arange(1, n_max + 1)    #                                       分子：出现次数 ↑               分母：总次数 ↑fig, ax = plt.subplots(figsize=(12, 5))for i in range(6):    ax.plot(range(1, n_max + 1), cumulative_freq[:, i],            linewidth=0.8, alpha=0.8, label=f'点数 {i+1}')ax.axhline(y=1/6, color='black', linestyle='--', linewidth=1.5, label='理论值 1/6')ax.set_xlabel('试验次数')ax.set_ylabel('累计频率')ax.set_title('频率收敛曲线 - 大数定律的直观展示', fontsize=13, fontweight='bold')ax.legend(loc='center right', ncol=2)ax.set_xlim(0, n_max)ax.set_ylim(0.10, 0.22)  # 放大看 0.10~0.22 区间plt.tight_layout()plt.show()# 观察：前几百次波动很大，越往后越稳定在 0.1667 附近

---## 3.7 蒙特卡洛实战 ②：估算 π 值**核心思想：** 在 2×2 的正方形内随机撒点，数有多少落在内切圆内。```    正方形面积 = 2 x 2 = 4    内切圆面积 = pi x 1^2 = pi        圆内点数 / 总点数 = 圆面积 / 正方形面积 = pi / 4        所以：pi = 4 x (圆内点数 / 总点数)```这就是蒙特卡洛的精髓：**用一个可以数出来的比例，去推算一个算不出来的数。**

In [ ]:
rng = np.random.default_rng(seed=42)n_points = 10_000# 在 [-1, 1] x [-1, 1] 正方形内随机撒点x = rng.uniform(-1, 1, n_points)  # 每个点的 x 坐标y = rng.uniform(-1, 1, n_points)  # 每个点的 y 坐标# 计算每个点到原点 (0, 0) 的距离# 勾股定理：距离 = sqrt(x^2 + y^2)distances = np.sqrt(x**2 + y**2)# 距离 <= 1 的点在圆内（圆半径为 1）inside = distances <= 1n_inside = np.sum(inside)  # True=1, False=0，求和 = 计数# 估算 pipi_estimate = 4 * n_inside / n_pointsprint(f"总撒点数:   {n_points:,}")print(f"圆内点数:   {n_inside:,}")print(f"圆内比例:   {n_inside / n_points:.4f}")print(f"pi 估计值:   {pi_estimate:.6f}")print(f"pi 真实值:   {np.pi:.6f}")print(f"相对误差:   {abs(pi_estimate - np.pi) / np.pi * 100:.4f}%")

In [ ]:
# ─── 可视化撒点结果 ───fig, axes = plt.subplots(1, 2, figsize=(14, 6))# 左图：撒点图（只画前 2000 个，避免太密看不清）n_show = min(2000, n_points)# 圆内的点用蓝色，圆外的点用红色axes[0].scatter(x[:n_show][inside[:n_show]], y[:n_show][inside[:n_show]],               c='steelblue', s=2, alpha=0.6, label='圆内点')axes[0].scatter(x[:n_show][~inside[:n_show]], y[:n_show][~inside[:n_show]],               c='salmon', s=2, alpha=0.6, label='圆外点')# ~inside 是取反：圆内的变圆外，圆外的变圆内# 画圆和正方形边界theta = np.linspace(0, 2*np.pi, 200)axes[0].plot(np.cos(theta), np.sin(theta), 'black', linewidth=2, label='单位圆')rect = plt.Rectangle((-1, -1), 2, 2, fill=False, edgecolor='black', linewidth=2, linestyle='--')axes[0].add_patch(rect)axes[0].set_aspect('equal')  # x 轴和 y 轴等比例，圆才不会变成椭圆axes[0].set_title(f'蒙特卡洛撒点 (显示前 {n_show} 个)', fontsize=13, fontweight='bold')axes[0].set_xlim(-1.1, 1.1)axes[0].set_ylim(-1.1, 1.1)axes[0].legend(loc='upper right', markerscale=3)# 右图：pi 估计值的收敛过程cumulative_inside = np.cumsum(inside)  # 累加：第 n 个位置 = 前 n 个点里有多少在圆内cumulative_pi = 4 * cumulative_inside / np.arange(1, n_points + 1)axes[1].plot(range(1, n_points + 1), cumulative_pi, linewidth=0.8, color='steelblue')axes[1].axhline(y=np.pi, color='red', linestyle='--', linewidth=1.5,                label=f'pi 真实值 = {np.pi:.6f}')axes[1].fill_between(range(1, n_points + 1), cumulative_pi, np.pi, alpha=0.15, color='red')axes[1].set_xlabel('撒点数量')axes[1].set_ylabel('pi 估计值')axes[1].set_title(f'pi 估计值的收敛过程 (最终: {pi_estimate:.4f})', fontsize=13, fontweight='bold')axes[1].legend()plt.tight_layout()plt.show()

### 不同样本量下的估算精度蒙特卡洛的收敛速度是 O(1/sqrt(n))：**想多一位精度，需要 100 倍的样本量。**

In [ ]:
rng = np.random.default_rng(seed=2024)sample_sizes = [100, 500, 1000, 5000, 10_000, 50_000, 100_000, 1_000_000]print(f"{'样本量':>12}  {'pi 估计值':>12}  {'误差':>12}  {'误差率':>10}")print("-" * 55)for n in sample_sizes:    x = rng.uniform(-1, 1, n)    y = rng.uniform(-1, 1, n)    pi_est = 4 * np.sum(x**2 + y**2 <= 1) / n    error = abs(pi_est - np.pi)    error_pct = error / np.pi * 100    print(f"{n:>12,}  {pi_est:>12.6f}  {error:>12.6f}  {error_pct:>9.4f}%")# 观察：100 个点可能差 0.2，100 万个点只差 0.000x

## 3.8 小结 & 自检清单| 技能 | ✓ ||------|---|| 理解「随机数其实是确定的」——种子的作用 | ☐ || 会用 `default_rng(seed)` 创建可复现的随机数生成器 | ☐ || `uniform`：生成均匀分布的随机数 | ☐ || `normal`：生成正态分布的随机数，理解 loc/scale 参数 | ☐ || `integers`：生成随机整数，注意 endpoint 参数 | ☐ || 理解蒙特卡洛思想：大量随机试验 → 逼近确定答案 | ☐ || 能用撒点法估算 π，理解收敛速度 | ☐ |### 关键公式```蒙特卡洛估算 pi = 4 x (圆内点数 / 总点数)正态分布 N(mu, sigma)：  68% 的数据在 mu +/- sigma 内  95% 的数据在 mu +/- 2*sigma 内```### 下一讲进入 **Pandas 时间序列分析**——用真实的金融数据来实战。---